In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import random
import cv2
import numpy as np
import torch
import albumentations as A
import ultralytics.data.build as ubuild
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
from ultralytics.utils.instance import Instances
from ultralytics.utils.ops import segments2boxes

# ---------------- A100 ayarları ----------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

NUM_WORKERS = min(os.cpu_count(), 12)
print("CPU çekirdek:", os.cpu_count(), "| workers:", NUM_WORKERS)

# ---------------- Ağırlıklı rotasyon ----------------
P_ZERO, P_NINETY = 0.60, 0.20      # kalan %20 → eğik
TILT_RANGE = (15, 75)

class RotYOLODataset(YOLODataset):
    def get_image_and_label(self, index):
        label = super().get_image_and_label(index)
        if self.augment:                       # sadece train setinde
            label = self._weighted_rotate(label)
        return label

    def _weighted_rotate(self, label):
        r = random.random()
        if r < P_ZERO:
            return label

        img = label["img"]
        h, w = img.shape[:2]
        segs = label["instances"].segments.copy()   # (N, M, 2), normalize

        if r < P_ZERO + P_NINETY:
            # Tam 90 / 270: kayıpsız
            k = random.choice([1, 3])
            img = np.ascontiguousarray(np.rot90(img, k))
            if len(segs):
                x, y = segs[..., 0].copy(), segs[..., 1].copy()
                if k == 1:   # saat yönü tersine
                    segs[..., 0], segs[..., 1] = y, 1 - x
                else:        # saat yönünde
                    segs[..., 0], segs[..., 1] = 1 - y, x
        else:
            # Eğik: tuvali genişlet, hiçbir kitap kesilmesin
            ang = random.uniform(*TILT_RANGE) * random.choice([-1, 1])
            M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
            cos, sin = abs(M[0, 0]), abs(M[0, 1])
            nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
            M[0, 2] += nw / 2 - w / 2
            M[1, 2] += nh / 2 - h / 2
            img = cv2.warpAffine(img, M, (nw, nh), borderValue=(114, 114, 114))
            if len(segs):
                pts = segs.reshape(-1, 2) * [w, h]
                pts = np.hstack([pts, np.ones((len(pts), 1))]) @ M.T
                segs = (pts / [nw, nh]).reshape(segs.shape)
            # Uzun kenarı tekrar imgsz'e indir (normalize koordinatlar değişmez)
            s = self.imgsz / max(nh, nw)
            if s < 1:
                img = cv2.resize(img, (round(nw * s), round(nh * s)),
                                 interpolation=cv2.INTER_LINEAR)

        segs = segs.astype(np.float32)
        bboxes = segments2boxes(segs) if len(segs) else np.zeros((0, 4), np.float32)
        label["img"] = img
        label["resized_shape"] = img.shape[:2]
        label["instances"] = Instances(bboxes, segs, None,
                                       bbox_format="xywh", normalized=True)
        return label

ubuild.YOLODataset = RotYOLODataset   # Ultralytics bizim sınıfı kullansın

# ---------------- Albumentations (piksel seviyesi) ----------------
custom_aug = [
    A.OneOf([
        A.MotionBlur(blur_limit=(3, 9)),
        A.Defocus(radius=(1, 3)),
        A.GaussianBlur(blur_limit=(3, 5)),
    ], p=0.25),
    A.OneOf([
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4)),
        A.GaussNoise(std_range=(0.02, 0.06)),
    ], p=0.2),
    A.ImageCompression(quality_range=(50, 95), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.1),
    A.RandomShadow(p=0.1),
]

# ---------------- Eğitim ----------------
model = YOLO("yolo26s-obb.pt")

model.train(
    data="/content/dataset/dataset/data.yaml",
    project="/content/drive/MyDrive/Bitirme/YOLO26_OBB",
    name="YOLO26s_OBB_1280_MuSGD_B64_wrot",

    # Donanım
    device=0,
    amp=True,
    batch=48,
    workers=NUM_WORKERS,
    cache="ram",
    imgsz=1280,

    # Eğitim
    epochs=200,
    patience=50,
    optimizer="MuSGD",
    seed=42,
    deterministic=False,
    plots=True,

    # Geometri
    degrees=5.0,
    flipud=0.5,
    fliplr=0.5,
    translate=0.1,
    scale=0.35,
    shear=2.0,
    perspective=0.0005,

    # Kompozisyon
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.0,
    copy_paste=0.0,

    # Renk
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.45,

    augmentations=custom_aug,
)

CPU çekirdek: 12 | workers: 12
Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.25, 'transforms': [{'__class_fullname__': 'MotionBlur', 'p': 0.5, 'allow_shifted': True, 'angle_range': (0.0, 360.0), 'blur_limit': (3, 9), 'direction_range': (-1.0, 1.0)}, {'__class_fullname__': 'Defocus', 'p': 0.5, 'alias_blur': (0.1, 0.5), 'radius': (1, 3)}, {'__class_fullname__': 'GaussianBlur', 'p': 0.5, 'blur_limit': (3, 5), 'sigma_limit': (0.5, 3.0)}]}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.2, 'transforms': [{'__class_fullname__': 'ISONoise', 'p': 0.5, 'color_shift': (0.01, 0.05), 'intensity': (0.1, 0.4)}, {'__class_fullname__': 'GaussNoise', 'p': 0.5, 'mean_range': (0.0, 0.0), 'noise_scale_factor': 1.0, 'per_channel': True, 'std_range': (0.02, 0.06)}]}

/usr/local/lib/python3.13/dist-packages/ultralytics/data/augment.py:1317: RuntimeWarning: divide by zero encountered in divide
  xy = xy[:, :2] / xy[:, 2:3]


     64/200      62.5G     0.9816     0.4549     0.1301   0.004733         72       1280: 100% ━━━━━━━━━━━━ 35/35 1.3it/s 26.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2it/s 8.1s
                   all        435       8599      0.964       0.98      0.984      0.796

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
     65/200      62.6G     0.9671     0.4549     0.1282   0.003808        114       1280: 100% ━━━━━━━━━━━━ 35/35 1.2it/s 29.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3it/s 7.7s
                   all        435       8599      0.961      0.983      0.983      0.789

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
     66/200      63.3G     0.9862     0.4641     0.1294   0.004434        112       1280: 100% ━━━━━━━━━━━━ 35/35 1.2it/s 29.5s
    

/usr/local/lib/python3.13/dist-packages/ultralytics/data/augment.py:1317: RuntimeWarning: divide by zero encountered in divide
  xy = xy[:, :2] / xy[:, 2:3]


    110/200      62.5G     0.8866     0.3929     0.1363   0.003234         29       1280: 100% ━━━━━━━━━━━━ 35/35 1.1it/s 31.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3it/s 7.8s
                   all        435       8599      0.975      0.986      0.988      0.848

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
    111/200      62.8G     0.9002     0.3872     0.1283   0.003234         94       1280: 100% ━━━━━━━━━━━━ 35/35 1.1it/s 33.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2it/s 8.4s
                   all        435       8599      0.976      0.981      0.988      0.848

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
    112/200      64.1G     0.8902     0.3808      0.129   0.003091         89       1280: 100% ━━━━━━━━━━━━ 35/35 1.2it/s 29.8s
    

ultralytics.utils.metrics.OBBMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79daf9b74050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 